импорты

In [7]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

загрузка датасета

In [16]:
df = pd.read_csv("../data/north.csv")
df.head()

,index,Data,Hora,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),...,"VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",region,state,station,station_code,latitude,longitude,height
0,0,2000-05-09,00:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,N,AM,MANAUS,A101,-3.103333,-60.016389,61.25
1,1,2000-05-09,01:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,N,AM,MANAUS,A101,-3.103333,-60.016389,61.25
2,2,2000-05-09,02:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,N,AM,MANAUS,A101,-3.103333,-60.016389,61.25
3,3,2000-05-09,03:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,N,AM,MANAUS,A101,-3.103333,-60.016389,61.25
4,4,2000-05-09,04:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,N,AM,MANAUS,A101,-3.103333,-60.016389,61.25


замена служебных пропусков

In [18]:
df = df.replace(-9999, np.nan)
df.isna().sum()

index                                                          0
Data                                                           0
Hora                                                           0
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                         1988221
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    1591222
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          1597892
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         1597871
RADIACAO GLOBAL (Kj/m²)                                  4540804
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)             1551208
TEMPERATURA DO PONTO DE ORVALHO (°C)                     1706311
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)               1558370
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)               1558241
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)         1713899
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)         1715709
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                 1707478
UMIDADE REL. MIN. NA HORA

переименование основных колонок

In [19]:
df = df.rename(columns={
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)": "temperature",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)": "pressure",
    "UMIDADE RELATIVA DO AR, HORARIA (%)": "humidity",
    "VENTO, VELOCIDADE HORARIA (m/s)": "wind_speed",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)": "precipitation"
})

df.columns

Index(['index', 'Data', 'Hora', 'precipitation', 'pressure',
       'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
       'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
       'RADIACAO GLOBAL (Kj/m²)', 'temperature',
       'TEMPERATURA DO PONTO DE ORVALHO (°C)',
       'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
       'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
       'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)', 'humidity',
       'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)',
       'wind_speed', 'region', 'state', 'station', 'station_code', 'latitude',
       'longitude', 'height'],
      dtype='object')

работа с датой и временем

In [20]:
df["Data"] = pd.to_datetime(df["Data"])

# берем данные начиная с 2010 года
df = df[df["Data"].dt.year >= 2010].copy()

# теперь заменяем служебные пропуски
df = df.replace(-9999, np.nan)

In [21]:
df["temperature"].isna().sum()

1447705

In [22]:
df.shape

(7395840, 27)

In [25]:
df["hour"] = df["Hora"].str[:2].astype(int)
df["year"] = df["Data"].dt.year
df["month"] = df["Data"].dt.month
df["day"] = df["Data"].dt.day

df[["Data", "Hora", "hour", "year", "month", "day"]].head()

,Data,Hora,hour,year,month,day
996480,2010-12-09,08:00,8,2010,12,9
996481,2010-12-09,09:00,9,2010,12,9
996482,2010-12-09,10:00,10,2010,12,9
996483,2010-12-09,11:00,11,2010,12,9
996484,2010-12-09,12:00,12,2010,12,9


выбор признаков и целевой переменной

In [26]:
feature_cols = [
    "hour",
    "month",
    "precipitation",
    "pressure",
    "humidity",
    "wind_speed",
    "latitude",
    "longitude",
    "height"
]

target_col = "temperature"

удалить строки без target

In [27]:
df_model = df.dropna(subset=[target_col]).copy()

X = df_model[feature_cols]
y = df_model[target_col]

X.head()

,hour,month,precipitation,pressure,humidity,wind_speed,latitude,longitude,height
996480,8,12,0.0,1001.5,100.0,0.9,-3.103333,-60.016389,61.25
996481,9,12,0.0,1001.6,99.0,1.1,-3.103333,-60.016389,61.25
996482,10,12,0.0,1002.4,100.0,0.3,-3.103333,-60.016389,61.25
996483,11,12,0.0,1003.4,94.0,0.9,-3.103333,-60.016389,61.25
996484,12,12,0.0,1003.5,82.0,1.9,-3.103333,-60.016389,61.25


размер данных

In [28]:
print("Количество объектов:", len(df_model))
print("Количество признаков:", X.shape[1])
print("Пропуски в X:")
print(X.isna().sum())

Количество объектов: 5948135
Количество признаков: 9
Пропуски в X:
hour                  0
month                 0
precipitation    440281
pressure          43027
humidity         155489
wind_speed       243881
latitude              0
longitude             0
height           101064
dtype: int64


разделение на train/test

In [36]:
df_model = df_model.sample(2000000, random_state=42)
print(len(df_model))

2000000


In [37]:
X = df_model[feature_cols]
y = df_model[target_col]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(1600000, 9) (400000, 9)


список числовых признаков

In [38]:
numeric_features = feature_cols
numeric_features

['hour',
 'month',
 'precipitation',
 'pressure',
 'humidity',
 'wind_speed',
 'latitude',
 'longitude',
 'height']

препроцессинг

In [39]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features)
])

базовый pipeline

In [40]:
pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge())
])

pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['hour', 'month',
                                                   'precipitation', 'pressure',
                                                   'humidity', 'wind_speed',
                                                   'latitude', 'longitude',
                                                   'height'])])),
                ('regressor', Ridge())])

сетка параметров

In [41]:
param_grid = {
    "regressor__alpha": np.linspace(0.1, 5.0, num=20)
}

GridSearchCV

In [42]:
search_cv = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

обучение

In [43]:
search_cv.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['hour',
                                                                          'month',
                                                                          'precipitation',
                                                                          'pressure',
                                                                          'humidity',
                                                                          'wind_speed',
                                                                          'latitude',
                                                                          'longitude',
                                                                          'height'])])),
                                       ('regressor', Ridge())]),
             n_jobs=-1,
             param_grid={'regressor__alpha': array([0.1       , 0.35789474, 0.61578947, 0.87368421, 1.13157895,
       1.38947368, 1.64736842, 1.90526316, 2.16315789, 2.42105263,
       2.67894737, 2.93684211, 3.19473684, 3.45263158, 3.71052632,
       3.96842105, 4.22631579, 4.48421053, 4.74210526, 5.        ])},
             scoring='r2')

лучшие параметры

In [44]:
print("Best params:")
print(search_cv.best_params_)

Best params:
{'regressor__alpha': 5.0}


качество на train

In [45]:
train_pred = search_cv.predict(X_train)

print("Train R2:", r2_score(y_train, train_pred))
print("Train MAE:", mean_absolute_error(y_train, train_pred))
print("Train RMSE:", mean_squared_error(y_train, train_pred, squared=False))

Train R2: 0.7287345046409087
Train MAE: 1.3726263352762171
Train RMSE: 1.9228579880078531


качество на test

In [46]:
test_pred = search_cv.predict(X_test)

print("Test R2:", r2_score(y_test, test_pred))
print("Test MAE:", mean_absolute_error(y_test, test_pred))
print("Test RMSE:", mean_squared_error(y_test, test_pred, squared=False))

Test R2: 0.7292123362297431
Test MAE: 1.3711406291570358
Test RMSE: 1.9205762153408608


несколько предсказаний

comparison = pd.DataFrame({
    "real_temperature": y_test.iloc[:10].values,
    "predicted_temperature": test_pred[:10]
})

comparison

сохранение модели

In [48]:
Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(search_cv.best_estimator_, "../models/weather_pipeline.pkl")
print("Model saved to ../models/weather_pipeline.pkl")

Model saved to ../models/weather_pipeline.pkl


проверка загрузки модели

In [49]:
loaded_model = joblib.load("../models/weather_pipeline.pkl")
loaded_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['hour', 'month',
                                                   'precipitation', 'pressure',
                                                   'humidity', 'wind_speed',
                                                   'latitude', 'longitude',
                                                   'height'])])),
                ('regressor', Ridge(alpha=5.0))])

тестовый инференс

In [51]:
sample = pd.DataFrame([{
    "hour": 12,
    "month": 6,
    "precipitation": 0.0,
    "pressure": 1008.0,
    "humidity": 75.0,
    "wind_speed": 3.5,
    "latitude": -3.1,
    "longitude": -60.0,
    "height": 61.25
}])

prediction = loaded_model.predict(sample)
print("Predicted temperature:", prediction[0])

Predicted temperature: 27.837863686043015
